[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/20.%20The%20SLAP%20for%20Out-of-Gauge%20%28OOG%29%20Containers%20Problem/P20-Tier-2_executed.ipynb)

---


# 20. The SLAP for Out-of-Gauge (OOG) Containers Problem

## Tier 2: The Classic Heuristic (Priority-Based OOG Assignment)

### Purpose and scope

This notebook assigns OOG containers to vessel slots using a priority-based greedy heuristic. It calculates a score for each feasible container-slot pair, chooses the pair with the highest score, updates the feasible candidate set, and repeats until every container is assigned or no feasible pair remains. The example also compares the heuristic with a Hungarian-algorithm assignment based on cost alone and examines how changing the priority weights affects the heuristic.

**Inputs:** container and slot characteristics, an assignment-cost matrix, stability scores, bay weight limits, and priority weights.

**Outputs:** assignments and their selection order, aggregate performance measures, a cost-only Hungarian reference, visualizations, and a priority-weight sensitivity analysis.

### Model assumptions

- Each container can be assigned to at most one slot, and each slot can hold at most one container.
- A container-slot pair is feasible only when the container satisfies the slot's weight and height limits and the resulting total weight in that slot's bay does not exceed the bay limit.
- The supplied cost and stability values are fixed for each pair. In this example, the priority components are also fixed: the set of feasible pairs changes after an assignment, but the score of a particular pair does not depend on earlier assignments.
- The stability matrix is a score to maximize, not a vessel-stability calculation. The reported lashing-complexity measure is a simple proxy, calculated as lashing coefficient multiplied by crane distance; it is not a physical lashing-cost model.
- The example omits operational details such as container orientation, neighboring-slot interactions, handling sequences, and uncertain or changing data. It is a small illustrative assignment problem, not a complete vessel-planning model.

### Heuristic workflow

1. Calculate the weighted priority score for every currently feasible container-slot pair.
2. Store candidates in a priority queue, with the highest score selected first.
3. Assign the selected container to the selected slot if the pair remains feasible.
4. Recheck the remaining candidates against the updated assignments and rebuild their queue priorities.
5. Summarize the resulting assignment and examine its cost, stability scores, capacity utilization, and bay loads.

The queue is dynamic because feasibility changes as assignments consume slots and bay capacity. The score formula itself uses static pair and container attributes in this example, so recalculating a surviving pair gives it the same score. This distinction is important when interpreting the selection sequence.

### Priority model

Let $C$ be the set of containers and $S$ the set of slots. For a candidate pair $(i,j)$, the score is a weighted sum of four normalized components:

$$
P_{ij} = w_1 U_{ij} + w_2 E_{ij} + w_3 T_{ij} + w_4 L_i.
$$

Here $U_{ij}=\min(q_i/Q_j,1)$ is the container's weight divided by the slot's weight capacity, capped at one; $E_{ij}=1-c_{ij}/c_{\max}$ rewards lower assignment cost; $T_{ij}$ is the supplied stability score; and $L_i=1-\ell_i/\ell_{\max}$ rewards a lower lashing coefficient. The weights $w_1,\ldots,w_4$ are nonnegative and sum to one in the supplied scenarios. In the code, these quantities are calculated in `calculate_priority_score` and stored in `score_components`.

The heuristic repeatedly chooses the feasible pair with maximum $P_{ij}$. Feasibility is checked separately from scoring: a high score cannot override a slot or bay constraint. In code, `priority_based_oog_assignment` implements this greedy selection, while `is_feasible_assignment` checks the constraints.

### What to examine

- Which container-slot pairs are selected, and how their score components contribute to their priorities.
- How the available candidate set changes as slots and bay capacity are consumed.
- Whether all containers are assigned while satisfying the stated constraints.
- How the heuristic's assignment cost compares with a cost-only assignment reference, and whether that reference satisfies the additional constraints.
- How the weight scenarios affect the heuristic's assignments and reported measures.

In [ ]:
# These libraries support the priority queue, problem data, numerical work, and plots.
import heapq
from dataclasses import dataclass, field
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported successfully for priority-based OOG assignment.")

In [ ]:
@dataclass
class OOGContainer:
    """An out-of-gauge container and its assignment-relevant attributes."""

    id: int
    weight: float  # tons
    height: float  # meters
    length: float  # meters
    container_type: str
    lashing_coeff: float
    time_sensitivity: float  # 0 to 1; retained as descriptive input
    handling_difficulty: float  # 0 to 1; retained as descriptive input


@dataclass
class VesselSlot:
    """An available vessel slot and the limits used to check feasibility."""

    id: int
    weight_capacity: float  # tons
    height_limit: float  # meters
    bay_id: int
    crane_distance: float
    slot_position: str


@dataclass
class PriorityAssignment:
    """A candidate pair, its priority score, and its score components."""

    priority: float
    container_id: int  # zero-based index in the problem's container list
    slot_id: int  # zero-based index in the problem's slot list
    score_components: Dict[str, float] = field(default_factory=dict)


@dataclass
class HeuristicProblem:
    """All data required by the assignment heuristic and its analysis."""

    containers: List[OOGContainer]
    slots: List[VesselSlot]
    cost_matrix: np.ndarray
    stability_matrix: np.ndarray
    bay_weight_limits: Dict[int, float]
    priority_weights: Dict[str, float]


print("Problem data structures are ready.")

In [ ]:
# The example data define three containers, four slots, and pair-specific scores.
containers = [
    OOGContainer(1, 45, 2.8, 12.0, "flat rack", 2.1, 0.7, 0.8),
    OOGContainer(2, 32, 4.2, 6.0, "open-top", 3.8, 0.9, 0.6),
    OOGContainer(3, 38, 2.6, 12.0, "platform", 1.9, 0.5, 0.4),
]

slots = [
    VesselSlot(1, 50, 3.0, 2, 15.0, "forward"),
    VesselSlot(2, 40, 5.0, 3, 12.0, "midship"),
    VesselSlot(3, 45, 3.0, 3, 18.0, "midship"),
    VesselSlot(4, 60, 4.0, 3, 10.0, "aft"),
]

# Rows correspond to containers in list order; columns correspond to slots.
cost_matrix = np.array(
    [
        [100, 150, 120, 90],
        [80, 70, 110, 85],
        [95, 140, 100, 80],
    ]
)

# These supplied scores are maximized by the priority rule.
stability_matrix = np.array(
    [
        [0.8, 0.7, 0.9, 0.85],
        [0.75, 0.8, 0.7, 0.9],
        [0.85, 0.65, 0.8, 0.75],
    ]
)

bay_weight_limits = {2: 80, 3: 120}

# The weights define the relative contribution of each priority component.
priority_weights = {
    "capacity_utilization": 0.30,
    "cost_efficiency": 0.25,
    "stability_contribution": 0.25,
    "lashing_simplicity": 0.20,
}

problem = HeuristicProblem(
    containers,
    slots,
    cost_matrix,
    stability_matrix,
    bay_weight_limits,
    priority_weights,
)

assert cost_matrix.shape == (len(containers), len(slots))
assert stability_matrix.shape == cost_matrix.shape
assert set(slot.bay_id for slot in slots).issubset(bay_weight_limits)
assert np.isclose(sum(priority_weights.values()), 1.0)
assert all(weight >= 0 for weight in priority_weights.values())

print(
    f"Problem defined: {len(containers)} containers and "
    f"{len(slots)} slots."
)
print(f"Priority weights: {priority_weights}")
print("Container characteristics:")
for container in containers:
    print(
        f"  C{container.id}: {container.container_type}, "
        f"{container.weight} tons, {container.height} m, "
        f"lashing coefficient {container.lashing_coeff}, "
        f"time sensitivity {container.time_sensitivity}, "
        f"handling difficulty {container.handling_difficulty}"
    )

## Priority scores and feasibility

A priority score ranks feasible candidates; it does not make an infeasible candidate acceptable. The feasibility test therefore checks each constraint before the pair can be selected. Container and slot arguments in the implementation are zero-based list indices, while the `id` fields are human-readable labels used in reports.

For a partial assignment $A$, a candidate $(i,j)$ must use an unoccupied slot, satisfy $q_i \leq Q_j$ and $h_i \leq H_j$, and keep the total assigned weight in slot $j$'s bay at or below that bay's limit. The code checks the weight and height conditions directly and sums the weights already assigned to the candidate slot's bay.

In [ ]:
def calculate_priority_score(
    problem: HeuristicProblem,
    container_id: int,
    slot_id: int,
    current_assignments: Dict[int, int],
) -> PriorityAssignment:
    """Calculate the weighted priority score for one container-slot pair."""
    container = problem.containers[container_id]
    slot = problem.slots[slot_id]

    # These normalized components make the weighted score interpretable.
    capacity_utilization = container.weight / slot.weight_capacity
    capacity_score = min(capacity_utilization, 1.0)

    max_cost = float(problem.cost_matrix.max())
    if max_cost <= 0:
        raise ValueError("The cost matrix must contain a positive maximum cost.")
    cost_score = 1.0 - (problem.cost_matrix[container_id, slot_id] / max_cost)

    stability_score = float(problem.stability_matrix[container_id, slot_id])

    max_lashing = max(item.lashing_coeff for item in problem.containers)
    if max_lashing <= 0:
        raise ValueError("Container lashing coefficients must have a positive maximum.")
    lashing_score = 1.0 - (container.lashing_coeff / max_lashing)

    priority = (
        problem.priority_weights["capacity_utilization"] * capacity_score
        + problem.priority_weights["cost_efficiency"] * cost_score
        + problem.priority_weights["stability_contribution"] * stability_score
        + problem.priority_weights["lashing_simplicity"] * lashing_score
    )

    score_components = {
        "capacity_utilization": capacity_score,
        "cost_efficiency": cost_score,
        "stability_contribution": stability_score,
        "lashing_simplicity": lashing_score,
        "priority": priority,
    }
    return PriorityAssignment(priority, container_id, slot_id, score_components)


def is_feasible_assignment(
    problem: HeuristicProblem,
    container_id: int,
    slot_id: int,
    current_assignments: Dict[int, int],
) -> bool:
    """Return whether the pair respects slot and bay constraints."""
    if not 0 <= container_id < len(problem.containers):
        return False
    if not 0 <= slot_id < len(problem.slots):
        return False
    if container_id in current_assignments:
        return False

    container = problem.containers[container_id]
    slot = problem.slots[slot_id]

    if slot_id in current_assignments.values():
        return False
    if container.weight > slot.weight_capacity:
        return False
    if container.height > slot.height_limit:
        return False

    # Bay capacity couples otherwise separate slot assignments.
    bay_weight = sum(
        problem.containers[assigned_container_id].weight
        for assigned_container_id, assigned_slot_id in current_assignments.items()
        if problem.slots[assigned_slot_id].bay_id == slot.bay_id
    )
    bay_weight += container.weight
    return bay_weight <= problem.bay_weight_limits[slot.bay_id]


print("Priority scoring and feasibility checks are defined.")

## Greedy assignment and solution measures

The priority queue is a min-heap in Python, so the implementation stores each score with a negative sign internally; this makes the largest actual priority come out first. Container and slot indices provide deterministic tie-breaking. After each assignment, the remaining candidates are checked again because bay loads and slot availability have changed.

For $n=|C|$ containers and $m=|S|$ slots, there can be at most $nm$ candidate pairs and at most $n$ successful assignment rounds. This implementation rebuilds the queue after each successful assignment. With a heap, its candidate processing is bounded on the order of $n^2m\log(nm)$ heap operations; feasibility checks also scan the current assignment, adding up to $O(n^3m)$ work in this direct implementation. These bounds describe the implemented procedure, not a guarantee of optimality. The greedy rule can return a partial assignment even when a different sequence could assign more containers.

In [ ]:
def priority_based_oog_assignment(
    problem: HeuristicProblem,
) -> Tuple[Dict[int, int], List[PriorityAssignment]]:
    """Assign containers iteratively using the highest feasible priority."""
    assignments: Dict[int, int] = {}
    assignment_history: List[PriorityAssignment] = []
    priority_queue = []

    # Negative scores adapt the max-priority rule to Python's min-heap.
    for container_id in range(len(problem.containers)):
        for slot_id in range(len(problem.slots)):
            if is_feasible_assignment(problem, container_id, slot_id, assignments):
                candidate = calculate_priority_score(
                    problem, container_id, slot_id, assignments
                )
                heapq.heappush(
                    priority_queue,
                    (-candidate.priority, container_id, slot_id, candidate),
                )

    while priority_queue and len(assignments) < len(problem.containers):
        _, _, _, current_assignment = heapq.heappop(priority_queue)

        if not is_feasible_assignment(
            problem,
            current_assignment.container_id,
            current_assignment.slot_id,
            assignments,
        ):
            continue

        assignments[current_assignment.container_id] = current_assignment.slot_id
        assignment_history.append(current_assignment)

        # Rebuild the queue so every surviving candidate reflects current feasibility.
        updated_queue = []
        while priority_queue:
            _, _, _, candidate = heapq.heappop(priority_queue)
            if is_feasible_assignment(
                problem,
                candidate.container_id,
                candidate.slot_id,
                assignments,
            ):
                updated_candidate = calculate_priority_score(
                    problem,
                    candidate.container_id,
                    candidate.slot_id,
                    assignments,
                )
                heapq.heappush(
                    updated_queue,
                    (
                        -updated_candidate.priority,
                        updated_candidate.container_id,
                        updated_candidate.slot_id,
                        updated_candidate,
                    ),
                )
        priority_queue = updated_queue

    return assignments, assignment_history


def analyze_heuristic_solution(
    problem: HeuristicProblem,
    assignments: Dict[int, int],
) -> Dict[str, object]:
    """Summarize assignment cost, supplied scores, and resource use."""
    total_cost = 0.0
    total_stability = 0.0
    total_capacity_used = 0.0
    total_capacity_available = 0.0
    lashing_complexity = 0.0
    bay_weights = {bay_id: 0.0 for bay_id in problem.bay_weight_limits}

    for container_id, slot_id in assignments.items():
        container = problem.containers[container_id]
        slot = problem.slots[slot_id]

        total_cost += float(problem.cost_matrix[container_id, slot_id])
        total_stability += float(problem.stability_matrix[container_id, slot_id])
        total_capacity_used += container.weight
        total_capacity_available += slot.weight_capacity
        lashing_complexity += container.lashing_coeff * slot.crane_distance
        bay_weights[slot.bay_id] += container.weight

    capacity_utilization = (
        100.0 * total_capacity_used / total_capacity_available
        if total_capacity_available
        else 0.0
    )
    return {
        "total_cost": total_cost,
        "total_stability": total_stability,
        "capacity_utilization": capacity_utilization,
        "lashing_complexity": lashing_complexity,
        "bay_weights": bay_weights,
        "assignments_count": len(assignments),
    }


print("Greedy assignment and solution-analysis functions are defined.")

In [ ]:
assignments, assignment_history = priority_based_oog_assignment(problem)
metrics = analyze_heuristic_solution(problem, assignments)

# Verify that the returned plan is internally consistent and feasible.
checked_assignments: Dict[int, int] = {}
for container_id, slot_id in assignments.items():
    assert is_feasible_assignment(problem, container_id, slot_id, checked_assignments)
    checked_assignments[container_id] = slot_id
assert len({slot_id for slot_id in assignments.values()}) == len(assignments)
assert metrics["assignments_count"] == len(assignment_history)

print("PRIORITY-BASED OOG ASSIGNMENT")
print("=" * 42)
print(
    f"Assignments made: {len(assignments)} of "
    f"{len(problem.containers)} containers"
)
print(f"Total assignment cost: {metrics['total_cost']:.1f} cost units")
print(f"Sum of supplied stability scores: {metrics['total_stability']:.2f}")
print(f"Assigned-slot weight utilization: {metrics['capacity_utilization']:.1f}%")
print(
    "Lashing-complexity proxy (coefficient × crane distance): "
    f"{metrics['lashing_complexity']:.1f}"
)

print("\nAssignment sequence (selection order):")
for step, assignment in enumerate(assignment_history, start=1):
    container = problem.containers[assignment.container_id]
    slot = problem.slots[assignment.slot_id]
    print(
        f"Step {step}: Container {container.id} → Slot {slot.id}; "
        f"priority {assignment.priority:.3f}"
    )
    print(
        f"  Container: {container.container_type}, {container.weight} tons, "
        f"{container.height} m"
    )
    print(
        f"  Slot: Bay {slot.bay_id}, {slot.weight_capacity} tons capacity, "
        f"{slot.height_limit} m height limit"
    )
    print("  Priority components:")
    for component, score in assignment.score_components.items():
        if component != "priority":
            print(f"    {component.replace('_', ' ').title()}: {score:.3f}")
    print(
        f"  Assignment cost: "
        f"{problem.cost_matrix[assignment.container_id, assignment.slot_id]}, "
        f"stability score: "
        f"{problem.stability_matrix[assignment.container_id, assignment.slot_id]:.2f}"
    )

print("Bay weight distribution:")
for bay_id, weight in metrics["bay_weights"].items():
    limit = problem.bay_weight_limits[bay_id]
    print(f"  Bay {bay_id}: {weight:.1f} of {limit} tons")

## Cost-only Hungarian reference

The Hungarian algorithm solves a linear assignment problem using the cost matrix alone. It does not include the height, individual slot-capacity, or bay-capacity constraints used by the heuristic. Its result is therefore a cost-only reference, not automatically an optimum for the full constrained problem. The code checks its feasibility explicitly before interpreting its cost as a lower-bound benchmark for feasible assignments.

The reference minimizes cost, whereas the heuristic ranks assignments using several weighted components. Stability and utilization differences are descriptive comparisons; they are not optimality gaps for those measures.

In [ ]:
from scipy.optimize import linear_sum_assignment

# This reference minimizes the supplied cost matrix without the extra constraints.
row_ind, col_ind = linear_sum_assignment(problem.cost_matrix)
optimal_assignments = {int(i): int(j) for i, j in zip(row_ind, col_ind)}
optimal_metrics = analyze_heuristic_solution(problem, optimal_assignments)

# Test feasibility in sequence, since each added load affects bay capacity.
reference_is_feasible = True
checked_reference: Dict[int, int] = {}
for container_id, slot_id in optimal_assignments.items():
    if not is_feasible_assignment(
        problem, container_id, slot_id, checked_reference
    ):
        reference_is_feasible = False
        break
    checked_reference[container_id] = slot_id

print("COST-ONLY HUNGARIAN REFERENCE AND PRIORITY HEURISTIC")
print("=" * 56)
print("Cost-only Hungarian reference:")
print(f"  Total cost: {optimal_metrics['total_cost']:.1f} cost units")
print(f"  Sum of supplied stability scores: {optimal_metrics['total_stability']:.2f}")
print(f"  Assigned-slot weight utilization: {optimal_metrics['capacity_utilization']:.1f}%")
print(f"  Feasible under the stated slot and bay constraints: {reference_is_feasible}")
print("Priority-based heuristic:")
print(f"  Total cost: {metrics['total_cost']:.1f} cost units")
print(f"  Sum of supplied stability scores: {metrics['total_stability']:.2f}")
print(f"  Assigned-slot weight utilization: {metrics['capacity_utilization']:.1f}%")

if optimal_metrics["total_cost"] != 0:
    cost_gap = (
        metrics["total_cost"] / optimal_metrics["total_cost"] - 1.0
    ) * 100.0
else:
    cost_gap = float("nan")

stability_difference = metrics["total_stability"] - optimal_metrics["total_stability"]
utilization_difference = (
    metrics["capacity_utilization"] - optimal_metrics["capacity_utilization"]
)

print("\nDescriptive comparison:")
if reference_is_feasible:
    print(f"  Heuristic cost relative to the cost-only reference: {cost_gap:+.1f}%")
else:
    print(
        "  The cost-only reference violates at least one stated constraint; "
        "its cost is not a feasible-plan benchmark."
    )
print(f"  Stability-score difference (heuristic minus reference): {stability_difference:+.2f}")
print(f"  Utilization difference (percentage points): {utilization_difference:+.1f}")

print("\nAssignments (container labels; reference → heuristic):")
for container_id, container in enumerate(problem.containers):
    reference_slot_id = optimal_assignments.get(container_id)
    heuristic_slot_id = assignments.get(container_id)
    reference_slot_label = (
        f"Slot {problem.slots[reference_slot_id].id}"
        if reference_slot_id is not None
        else "Unassigned"
    )
    heuristic_slot_label = (
        f"Slot {problem.slots[heuristic_slot_id].id}"
        if heuristic_slot_id is not None
        else "Unassigned"
    )
    print(f"  Container {container.id}: {reference_slot_label} → {heuristic_slot_label}")

    if reference_slot_id is not None and heuristic_slot_id is not None:
        cost_difference = (
            problem.cost_matrix[container_id, heuristic_slot_id]
            - problem.cost_matrix[container_id, reference_slot_id]
        )
        if cost_difference != 0:
            print(f"    Pair-cost difference: {cost_difference:+.0f} cost units")

## Visual analysis

The plots show four complementary views: the selected pairs' priority components, aggregate measures for the heuristic and cost-only reference, the sequence of selected priorities and pair costs, and bay loads relative to their limits. The reference is labeled as cost-only because it does not optimize stability or utilization and may not satisfy all constraints. The charts report measures; they do not establish that one assignment is best on every objective.

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Priority-Based OOG Assignment Analysis", fontsize=16, fontweight="bold")

# Priority components make the weighted ranking interpretable by selected pair.
priority_data = [
    [
        assignment.score_components["capacity_utilization"],
        assignment.score_components["cost_efficiency"],
        assignment.score_components["stability_contribution"],
        assignment.score_components["lashing_simplicity"],
    ]
    for assignment in assignment_history
]
labels = [
    f"C{problem.containers[a.container_id].id}→S{problem.slots[a.slot_id].id}"
    for a in assignment_history
]
components_names = ["Capacity", "Cost", "Stability", "Lashing"]
colors = ["skyblue", "lightgreen", "lightcoral", "gold"]

if priority_data:
    priority_array = np.asarray(priority_data).T
    x = np.arange(len(labels))
    width = 0.2
    for component_index, (name, color) in enumerate(zip(components_names, colors)):
        ax1.bar(
            x + component_index * width,
            priority_array[component_index],
            width,
            label=name,
            alpha=0.8,
            color=color,
        )
    ax1.set_xticks(x + width * 1.5)
    ax1.set_xticklabels(labels, rotation=45, ha="right")
    ax1.legend(title="Score component")
else:
    ax1.text(0.5, 0.5, "No assignments", ha="center", va="center")
ax1.set_xlabel("Selected container-slot pair")
ax1.set_ylabel("Component score")
ax1.set_title("Priority Components by Assignment")
ax1.grid(True, alpha=0.3)

# The reference measures are descriptive; only its objective is cost.
comparison_metrics = ["Total cost", "Stability-score sum", "Weight utilization (%)"]
reference_values = [
    optimal_metrics["total_cost"],
    optimal_metrics["total_stability"],
    optimal_metrics["capacity_utilization"],
]
heuristic_values = [
    metrics["total_cost"],
    metrics["total_stability"],
    metrics["capacity_utilization"],
]
x = np.arange(len(comparison_metrics))
width = 0.35
ax2.bar(x - width / 2, reference_values, width, label="Cost-only reference", color="blue", alpha=0.8)
ax2.bar(x + width / 2, heuristic_values, width, label="Priority heuristic", color="orange", alpha=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels(comparison_metrics, rotation=15, ha="right")
ax2.set_ylabel("Measure value")
ax2.set_title("Reference and Heuristic Measures")
ax2.legend()
ax2.grid(True, alpha=0.3)

# Each marker is a selection step; bars show that pair's assignment cost.
steps = np.arange(1, len(assignment_history) + 1)
priorities = [assignment.priority for assignment in assignment_history]
costs = [
    problem.cost_matrix[assignment.container_id, assignment.slot_id]
    for assignment in assignment_history
]
ax3_twin = ax3.twinx()
ax3.plot(steps, priorities, "o-", color="purple", linewidth=2, label="Priority score")
ax3_twin.bar(steps, costs, alpha=0.3, color="red", label="Pair assignment cost")
ax3.set_xlabel("Assignment step")
ax3.set_ylabel("Priority score", color="purple")
ax3_twin.set_ylabel("Pair assignment cost (units)", color="red")
ax3.set_title("Selection Sequence")
ax3.tick_params(axis="y", labelcolor="purple")
ax3_twin.tick_params(axis="y", labelcolor="red")
ax3.grid(True, alpha=0.3)
for step, assignment in zip(steps, assignment_history):
    container_label = problem.containers[assignment.container_id].id
    ax3.annotate(
        f"C{container_label}",
        (step, assignment.priority),
        xytext=(0, 7),
        textcoords="offset points",
        ha="center",
    )

# Bay loads are displayed against the limits used in the feasibility test.
bay_ids = list(problem.bay_weight_limits)
reference_bay_weights = [optimal_metrics["bay_weights"][bay_id] for bay_id in bay_ids]
heuristic_bay_weights = [metrics["bay_weights"][bay_id] for bay_id in bay_ids]
bay_limits = [problem.bay_weight_limits[bay_id] for bay_id in bay_ids]
x = np.arange(len(bay_ids))
width = 0.25
ax4.bar(x - width, bay_limits, width, label="Bay limit", color="gray", alpha=0.3)
ax4.bar(x, reference_bay_weights, width, label="Cost-only reference", color="blue", alpha=0.8)
ax4.bar(x + width, heuristic_bay_weights, width, label="Priority heuristic", color="orange", alpha=0.8)
ax4.set_xticks(x)
ax4.set_xticklabels([f"Bay {bay_id}" for bay_id in bay_ids])
ax4.set_xlabel("Bay")
ax4.set_ylabel("Assigned weight (tons)")
ax4.set_title("Bay Loads and Limits")
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("The assignment analysis plots are ready.")

## Sensitivity to priority weights

A weighted score expresses how strongly the heuristic values each component. Sensitivity analysis reruns the same assignment procedure on the same containers, slots, costs, stability scores, and bay limits, changing only the priority weights. It can reveal whether the greedy selection sequence and resulting measures depend on those preferences. It does not identify universally best weights: choosing weights requires operational judgment, and the reported measures should be interpreted together with feasibility and the intended priorities.

In [ ]:
weight_scenarios = [
    {
        "name": "Balanced (Base)",
        "weights": {
            "capacity_utilization": 0.30,
            "cost_efficiency": 0.25,
            "stability_contribution": 0.25,
            "lashing_simplicity": 0.20,
        },
    },
    {
        "name": "Cost-Focused",
        "weights": {
            "capacity_utilization": 0.20,
            "cost_efficiency": 0.40,
            "stability_contribution": 0.20,
            "lashing_simplicity": 0.20,
        },
    },
    {
        "name": "Stability-Focused",
        "weights": {
            "capacity_utilization": 0.25,
            "cost_efficiency": 0.20,
            "stability_contribution": 0.40,
            "lashing_simplicity": 0.15,
        },
    },
    {
        "name": "Capacity-Focused",
        "weights": {
            "capacity_utilization": 0.50,
            "cost_efficiency": 0.20,
            "stability_contribution": 0.20,
            "lashing_simplicity": 0.10,
        },
    },
]

results = []
for scenario in weight_scenarios:
    assert np.isclose(sum(scenario["weights"].values()), 1.0)
    modified_problem = HeuristicProblem(
        containers=problem.containers,
        slots=problem.slots,
        cost_matrix=problem.cost_matrix,
        stability_matrix=problem.stability_matrix,
        bay_weight_limits=problem.bay_weight_limits,
        priority_weights=scenario["weights"],
    )
    scenario_assignments, _ = priority_based_oog_assignment(modified_problem)
    scenario_metrics = analyze_heuristic_solution(modified_problem, scenario_assignments)
    results.append(
        {
            "scenario": scenario["name"],
            "cost": scenario_metrics["total_cost"],
            "stability": scenario_metrics["total_stability"],
            "utilization": scenario_metrics["capacity_utilization"],
            "assignments": scenario_metrics["assignments_count"],
        }
    )
    print(f"\n{scenario['name']}:")
    print(f"  Weights: {scenario['weights']}")
    print(f"  Total assignment cost: {scenario_metrics['total_cost']:.1f} cost units")
    print(f"  Sum of supplied stability scores: {scenario_metrics['total_stability']:.2f}")
    print(f"  Assigned-slot weight utilization: {scenario_metrics['capacity_utilization']:.1f}%")
    print(
        f"  Assignments: {scenario_metrics['assignments_count']} "
        f"of {len(problem.containers)} containers"
    )

# Plot the measures calculated for each weight scenario.
scenario_names = [result["scenario"] for result in results]
costs = [result["cost"] for result in results]
stabilities = [result["stability"] for result in results]
utilizations = [result["utilization"] for result in results]
assignment_counts = [result["assignments"] for result in results]

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("Priority-Weight Sensitivity Analysis", fontsize=16, fontweight="bold")

ax1.bar(scenario_names, costs, color="lightblue", alpha=0.8)
ax1.set_ylabel("Total assignment cost (units)")
ax1.set_title("Cost by Weight Scenario")
ax1.tick_params(axis="x", rotation=35)
for index, value in enumerate(costs):
    ax1.text(index, value, f"{value:.0f}", ha="center", va="bottom")

ax2.bar(scenario_names, stabilities, color="lightgreen", alpha=0.8)
ax2.set_ylabel("Sum of stability scores")
ax2.set_title("Stability Scores by Weight Scenario")
ax2.tick_params(axis="x", rotation=35)
for index, value in enumerate(stabilities):
    ax2.text(index, value, f"{value:.2f}", ha="center", va="bottom")

ax3.bar(scenario_names, utilizations, color="lightcoral", alpha=0.8)
ax3.set_ylabel("Assigned-slot weight utilization (%)")
ax3.set_title("Weight Utilization by Scenario")
ax3.tick_params(axis="x", rotation=35)
for index, value in enumerate(utilizations):
    ax3.text(index, value, f"{value:.1f}%", ha="center", va="bottom")

ax4.bar(scenario_names, assignment_counts, color="gold", alpha=0.8)
ax4.set_ylabel("Number of containers assigned")
ax4.set_title("Assignment Count by Scenario")
ax4.tick_params(axis="x", rotation=35)
ax4.set_ylim(0, len(problem.containers) + 0.5)
for index, value in enumerate(assignment_counts):
    ax4.text(index, value, f"{value}/{len(problem.containers)}", ha="center", va="bottom")

for axis in (ax1, ax2, ax3, ax4):
    axis.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print(
    "Interpret the scenario plots by comparing their measures and assignment counts. "
    "A change in one measure may accompany a change in another; the weights encode "
    "preferences rather than guarantees of a particular outcome."
)

## Role and limitations of the priority heuristic

The priority-based method is useful when an understandable sequence of local decisions is valuable and assignments must be produced through repeated feasibility checks. Its priority score can combine cost, capacity utilization, supplied stability scores, and lashing simplicity in a single ranking. The queue and feasibility checks make it possible to respond to the changing set of available pairs as assignments are made.

The method is greedy: each selection is made using the current candidate scores, without searching all possible complete assignments. It therefore has no general guarantee of finding a globally optimal assignment or assigning the largest possible number of containers. The weight choices affect the ranking, and the example's fixed score components do not model every operational interaction. A cost-only Hungarian assignment is useful as a reference for the cost matrix, but it is not a substitute for checking the additional constraints or for a full multi-objective model.

Use the results as a transparent demonstration of priority-based assignment and its trade-offs. Operational use requires validating the data, the feasibility rules, the score normalization, and the priority weights against the vessel and terminal's actual planning requirements.